In [ ]:
import os 

os.chdir("..")

In [ ]:
import polars as pl
from fsdc_calories import DataCal
import plotly.graph_objects as go
import pandas as pd 
dc = DataCal()

In [ ]:
df = dc.gen_nuti_data()
df = df.group_by(["fiscal_year"]).agg(pl.col("total_calories", "protein", "vegetable", "dairy", "fruits", "grains", "ice_cream", "fats", "sugars", "other").sum())
df =  df.with_columns(
            protein=pl.col("protein")/ pl.col("total_calories"),
            vegetable=pl.col("protein")/ pl.col("total_calories"),
            dairy=pl.col("dairy")/ pl.col("total_calories"),
            fruits=pl.col("fruits")/ pl.col("total_calories"),
            grains=pl.col("grains")/ pl.col("total_calories"),
            ice_cream=pl.col("ice_cream")/ pl.col("total_calories"),
            fats=pl.col("fats")/ pl.col("total_calories"),
            sugars=pl.col("sugars")/ pl.col("total_calories"),
            other=pl.col("other")/ pl.col("total_calories"),
        )
df.sort(["fiscal_year"])

In [ ]:
dc.gen_graphs_price_change()

In [ ]:
dc.gen_graphs_price_change()

In [ ]:
import plotly.graph_objects as go
import plotly.express as px

# Convert Polars DataFrame to Pandas for Plotly processing
pdf = df.to_pandas()
    
# Define categories to plot and clean them up for presentation (Title Case)
categories = ["protein", "vegetable", "dairy", "fruits", "grains", "other"]
category_labels = [cat.capitalize() for cat in categories]

# Sort the fiscal years in descending order (inverted sort)
fiscal_years = sorted(pdf["fiscal_year"].unique().tolist(), reverse=True)

# Generate a professional, color-blind friendly Viridis color palette mapped to categories
viridis_colors = px.colors.sample_colorscale("Viridis", len(categories))

fig = go.Figure()

# Add traces for each fiscal year
for i, year in enumerate(fiscal_years):
    row = pdf[pdf["fiscal_year"] == year]
    values = [row[cat].values[0] for cat in categories]
    
    fig.add_trace(
        go.Pie(
            labels=category_labels,
            values=values,
            name=str(year),
            visible=(i == 0),  # Show only the first year by default
            textinfo="label+percent",
            hoverinfo="label+percent+value",
            hovertemplate="<b>%{label}</b><br>Share: %{percent}<extra></extra>",
            marker=dict(
                colors=viridis_colors,
                line=dict(color="#FFFFFF", width=2)  # Crisp separation borders
            ),
            # Pull out the largest slice slightly for a modern aesthetic focal point
            pull=[0.03 if j == values.index(max(values)) else 0 for j in range(len(values))]
        )
    )
    
# Create robust dropdown buttons
buttons = []
for i, year in enumerate(fiscal_years):
    visibility = [False] * len(fiscal_years)
    visibility[i] = True
    
    buttons.append(
        dict(
            label=str(year),
            method="restyle",  # Use 'restyle' instead of 'update' for targeted trace visibility switching
            args=[
                {"visible": visibility},
                {"title.text": f"<b>Nutritional Proportion Breakdown — Fiscal Year {year}</b>"}
            ]
        )
    )
    
# Update layout to publication-ready standards
fig.update_layout(
    updatemenus=[
        dict(
            active=0,
            buttons=buttons,
            x=0.0,
            xanchor="left",
            y=1.15,
            yanchor="top",
            bgcolor="#F8F9FA",
            bordercolor="#CED4DA",
            borderwidth=1,
            pad={"r": 10, "t": 10, "b": 10, "l": 10},
            font=dict(family="Arial, sans-serif", size=12, color="#212529")
        )
    ],
    title=dict(
        text=f"<b>Puerto Rico Food Pate Proportion Breakdown — Fiscal Year {fiscal_years[0]}</b><br><sup>Comparative annual distribution across major food groups from imported data</sup>",
        font=dict(family="Arial, sans-serif", size=16, color="#212529"),
        x=0.0,  # Left-align title for a modern corporate/editorial look
        xanchor="left",
        y=0.95
    ),
    template="simple_white",
    font=dict(family="Arial, sans-serif", size=12, color="#212529"),
    legend=dict(
        title=dict(text="<b>Food Categories</b>"),
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.02,
        bgcolor="rgba(255,255,255,0.8)",
        bordercolor="#E9ECEF",
        borderwidth=1
    ),
    margin=dict(t=140, b=50, l=50, r=150),
    hoverlabel=dict(
        bgcolor="white",
        font_size=12,
        font_family="Arial, sans-serif"
    )
)

# Enable responsive resizing configuration for web/dashboard embedding
fig.show(config={"responsive": True, "displayModeBar": True})